# Mass generation — holdout Claude (OpenRouter)

**Только test** по `v2/docs/dataset_design_final.md` §6.9: `split="test"`. **Не** включать в train/val.

- **Полная сетка** `(family, subtype, bin, idx)` — независимо от доли OpenAI/Mistral. **`SAMPLES_PER_SUBTYPE` = 400** как в ноутбуках 11/12 → **9200** заданий на полный прогон.
- **Output:** `v2/data/interim/llm-generated/core_llm_holdout_claude_<model_slug>.jsonl`.
- **Resume:** append-only; существующие `generation_job_id` пропускаются.

**Скорость:** параллельные запросы — сначала **`LLM_GEN_MAX_WORKERS_HOLDOUT`**, иначе **`LLM_GEN_MAX_WORKERS`**; если ни одна не задана, в коде ниже по умолчанию **12** воркеров (выше, чем типичные 8 у seen, чтобы быстрее пройти полную сетку 9200). При 429 уменьшите.

**Env:** **`OPENROUTER_API_KEY_CLAUDE`** в `v2/.env` или корневом `.env` — отдельный ключ от Mistral. См. `v2/.env.example`.

**Security:** при утечке ключа — **rotate** в кабинете OpenRouter.

**Прогресс:** `tqdm` + postfix (см. `v2/src/llm_mass_generation.py`).

In [5]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

V2 = None
cur = Path.cwd().resolve()
for _ in range(24):
    if (cur / "config.py").is_file() and (cur / "src" / "llm_mass_generation.py").is_file():
        V2 = cur
        break
    nested = cur / "v2"
    if (nested / "config.py").is_file() and (nested / "src" / "llm_mass_generation.py").is_file():
        V2 = nested
        break
    if cur.parent == cur:
        break
    cur = cur.parent
if V2 is None:
    raise RuntimeError(
        "Не найден v2/: открой папку репозитория или v2/, либо смени cwd ядра Jupyter."
    )

sys.path.insert(0, str(V2))

from src.llm_mass_generation import MassGenConfig, run_mass_generation

load_dotenv(V2.parent / ".env")
load_dotenv(V2 / ".env")

# Must match notebooks 11 & 12 (400 × 23 = 9200 jobs).
SAMPLES_PER_SUBTYPE = 400
_hw = os.environ.get("LLM_GEN_MAX_WORKERS_HOLDOUT", "").strip()
_gw = os.environ.get("LLM_GEN_MAX_WORKERS", "").strip()
if _hw:
    MAX_WORKERS = max(1, int(_hw))
elif _gw:
    MAX_WORKERS = max(1, int(_gw))
else:
    MAX_WORKERS = 12  # holdout default: higher than seen (8) if no env
# OpenRouter slug — см. https://openrouter.ai/models (старые id с датой, напр. claude-3-5-haiku-20241022, дают 404).
MODEL = "anthropic/claude-3.5-haiku"

cfg = MassGenConfig(
    base=V2,
    lane="holdout_claude",
    api_key_env="OPENROUTER_API_KEY_CLAUDE",
    model=MODEL,
    origin_model=MODEL,
    split="test",
    samples_per_subtype=SAMPLES_PER_SUBTYPE,
    openai_base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "https://local.invalid/v2-core-llm",
        "X-Title": "v2 Core LLM holdout (Claude via OpenRouter)",
    },
    max_workers=MAX_WORKERS,
)

run_mass_generation(cfg)

Output:        /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_holdout_claude_anthropic_claude-3_5-haiku.jsonl
Lane:          holdout_claude
Model:         anthropic/claude-3.5-haiku
origin_model:  anthropic/claude-3.5-haiku
split:         test
max_workers:   8
Jobs total:    9200
Already done:  8841
Pending:       359


holdout_claude:   0%|          | 0/359 [00:00<?, ?job/s] 

    [QC fail 1/3] holdout_claude||fraud_sms_deceptive||account_alert||short||302: ['markdown_emphasis']


    [QC fail 1/3] holdout_claude||fraud_sms_deceptive||account_alert||short||1: ['markdown_emphasis']


    [QC fail 1/3] holdout_claude||fraud_sms_deceptive||account_alert||medium||281: ['markdown_emphasis']


    [QC fail 1/3] holdout_claude||fraud_sms_deceptive||account_alert||medium||226: ['markdown_emphasis']


    [QC fail 1/3] holdout_claude||fraud_sms_deceptive||account_alert||medium||61: ['markdown_emphasis']


    [near-dup 1/3] holdout_claude||fraud_sms_deceptive||account_alert||medium||183


    [QC fail 2/3] holdout_claude||fraud_sms_deceptive||account_alert||medium||61: ['markdown_emphasis']


    [QC fail 2/3] holdout_claude||fraud_sms_deceptive||account_alert||medium||183: ['markdown_emphasis']


    [near-dup 1/3] holdout_claude||fraud_sms_deceptive||account_alert||medium||87


    [near-dup 2/3] holdout_claude||fraud_sms_deceptive||account_alert||short||302


    

PosixPath('/Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_holdout_claude_anthropic_claude-3_5-haiku.jsonl')